# Shape × Scale: movie-level non-homogeneous Poisson

**Date:** 2026-04-18.
**Motivation:** Per-critic KDE has an architectural ceiling (Path B-lite §8). Ridge regression beats KDE on cohort MAE and calibration but has high per-target variance. A more principled alternative:

```
λ(t; target) = V(target) × g(t; cluster)
```

**Shape × Scale factorization:**
- `g(t; cluster)` is a NORMALIZED arrival-shape (cumulative fraction by time-since-first-review). Cohort-informed via combined_score-weighted average of training movies' empirical cumulatives.
- `V(target)` is target's expected TOTAL reviews. Inferred from observed data: `V = observed / F(obs_end)`.
- Prediction: `expected_future = V × (F(pred_end) − F(obs_end))`.

**Why this is more intuitive than per-critic KDE:**
1. Decouples "what shape" (which cluster of similar arrival patterns) from "how big" (total volume).
2. Observed data directly infers V — no awkward observed-critics exclusion.
3. Late-surge movies are a shape-cluster problem, not a base_rate problem.
4. Simpler: no per-critic KDE, no base_rate × KDE × exclusion chain.

**Test:** train on cohort minus 5 h/m movies, test on h/m holdout. 5-fold CV on cohort. Midnight+noon convention. Compare to weighted-KDE and Ridge.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name != 'notebooks':
    ROOT = ROOT / 'notebooks'
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold

from _helpers import (
    reviews, close_date_map, gap_lookup, first_review_ts, gap_for_slug,
    combined_score_with_scores,
    build_weighted_critic_profiles, build_weighted_kde_lambda_model,
    predict_window_custom,
    CACHE_DIR,
)

SHIP_ALPHA = 0.5
SHIP_SIGMA_GAP = 8.0
SHIP_N_TRAINING = 20
SNAP_DAYS = 3

# Noon-shifted reviews (midnight+noon convention)
reviews_noon = reviews.copy()
day_mask = reviews_noon['timestamp_confidence'] == 'd'
reviews_noon.loc[day_mask, 'estimated_timestamp'] = (
    reviews_noon.loc[day_mask, 'estimated_timestamp'] + pd.Timedelta(hours=12)
)
first_review_ts_noon = (reviews_noon[reviews_noon['movie_slug'].isin(close_date_map)]
                         .groupby('movie_slug')['estimated_timestamp'].min())

HM = ['the_drama', 'the_super_mario_galaxy_movie', 'forbidden_fruits_2026',
      'they_will_kill_you', 'you_me_and_tuscany']

CACHE = CACHE_DIR / 'shape_scale.pkl'
print('Ready.')

## Per-movie cumulative fraction function

For each movie, compute `F_movie(t)` = fraction of its reviews arrived by `t` days after its first review (using noon-shifted reviews for midnight+noon consistency).

In [ ]:
# Precompute per-movie arrival time offsets (days since first review, noon-shifted)
movie_arrivals = {}
for slug in close_date_map:
    mr = reviews_noon[reviews_noon['movie_slug'] == slug]
    if slug not in first_review_ts_noon.index:
        continue
    first_ts = first_review_ts_noon.loc[slug]
    close_ts = close_date_map[slug]
    # Only consider reviews before close
    mr = mr[mr['estimated_timestamp'] < close_ts]
    if len(mr) == 0:
        continue
    # Days since first review
    deltas = (mr['estimated_timestamp'] - first_ts).dt.total_seconds() / 86400
    # Sort
    deltas_sorted = np.sort(deltas.values)
    movie_arrivals[slug] = deltas_sorted

print(f'Computed arrivals for {len(movie_arrivals)} movies')
print(f'Arrivals count distribution: {pd.Series([len(v) for v in movie_arrivals.values()]).describe().round(1).to_dict()}')


def F_movie(slug, t):
    """Fraction of movie slug's reviews arrived by time t (days since first review)."""
    arrivals = movie_arrivals.get(slug)
    if arrivals is None or len(arrivals) == 0:
        return 0.0
    return float(np.searchsorted(arrivals, t, side='right') / len(arrivals))


def F_cluster(scores, t):
    """Weighted-average cumulative fraction across training movies at time t."""
    if len(scores) == 0:
        return 0.0
    total_w = sum(scores.values())
    if total_w <= 0:
        weights = {s: 1.0 / len(scores) for s in scores}
    else:
        weights = {s: w / total_w for s, w in scores.items()}
    return sum(weights[s] * F_movie(s, t) for s in scores)


## Visualize cohort-level cumulative shape for `the_drama`

Plot the weighted-average shape against each training movie's shape.

In [ ]:
target = 'the_drama'
target_gap = gap_for_slug(target)
target_close = close_date_map[target]
snap_time = target_close.floor('D') - pd.Timedelta(days=SNAP_DAYS)
snap_dbc_eff = (target_close - snap_time).total_seconds() / 86400
midnight_utc_dbc = (target_close - target_close.floor('D')).total_seconds() / 86400

obs = reviews_noon[(reviews_noon['movie_slug']==target) & (reviews_noon['estimated_timestamp']<snap_time)]
state = {
    'observed_critics': set(obs['reviewer_name']),
    'observed_count': len(obs),
    'first_review_dbc': float((target_close - obs['estimated_timestamp'].min()).total_seconds() / 86400),
}
target_window_days = state['first_review_dbc'] - snap_dbc_eff

scores = combined_score_with_scores(
    target, target_gap, state['observed_critics'], target_window_days,
    k=SHIP_N_TRAINING, alpha=SHIP_ALPHA, sigma_gap=SHIP_SIGMA_GAP,
)

# Plot
ts = np.linspace(0, 15, 300)
fig, ax = plt.subplots(figsize=(10, 5))
for slug in scores:
    F_vals = [F_movie(slug, t) for t in ts]
    ax.plot(ts, F_vals, color='lightgray', alpha=0.7, linewidth=0.8)
F_cluster_vals = [F_cluster(scores, t) for t in ts]
ax.plot(ts, F_cluster_vals, color='red', linewidth=2.5, label='weighted cluster shape')
# Target's own cumulative if we had perfect info
F_target_vals = [F_movie(target, t) for t in ts]
ax.plot(ts, F_target_vals, color='blue', linewidth=2, linestyle='--', label='target actual (oracle)')
# Vertical lines for snap and prediction end
ax.axvline(target_window_days, color='gray', linestyle=':', label=f'obs end (t={target_window_days:.2f})')
ax.axvline(state['first_review_dbc'] - midnight_utc_dbc, color='black', linestyle=':', label=f'pred end (t={state["first_review_dbc"]-midnight_utc_dbc:.2f})')
ax.set_xlabel('Days since first review')
ax.set_ylabel('Cumulative fraction of reviews')
ax.set_title(f'Shape model for {target} — cohort-weighted cluster vs target actual')
ax.legend()
ax.set_xlim(0, 10)
plt.tight_layout()
plot_path = ROOT.parent / 'notebooks' / 'shape_scale_the_drama.png' if ROOT.name == 'notebooks' else ROOT / 'notebooks' / 'shape_scale_the_drama.png'
plt.savefig(plot_path, dpi=120, bbox_inches='tight')
plt.show()
print(f'Saved {plot_path}')

## Predict phase_1 using shape × scale

In [ ]:
def shape_scale_predict(slug):
    """Return (prediction, actual, V_inferred) under midnight+noon convention."""
    target_close = close_date_map[slug]
    midnight_utc_dbc = (target_close - target_close.floor('D')).total_seconds() / 86400
    snap_time = target_close.floor('D') - pd.Timedelta(days=SNAP_DAYS)
    snap_dbc_eff = (target_close - snap_time).total_seconds() / 86400

    obs = reviews_noon[(reviews_noon['movie_slug']==slug) & (reviews_noon['estimated_timestamp']<snap_time)]
    if len(obs) < 3:
        return None
    target_gap = gap_for_slug(slug)
    if target_gap is None:
        return None
    state = {
        'observed_critics': set(obs['reviewer_name']),
        'observed_count': len(obs),
        'first_review_dbc': float((target_close - obs['estimated_timestamp'].min()).total_seconds() / 86400),
    }
    if state['first_review_dbc'] < snap_dbc_eff + 1.0:
        return None
    target_window_days = state['first_review_dbc'] - snap_dbc_eff
    scores = combined_score_with_scores(
        slug, target_gap, state['observed_critics'], target_window_days,
        k=SHIP_N_TRAINING, alpha=SHIP_ALPHA, sigma_gap=SHIP_SIGMA_GAP,
    )
    if len(scores) < 5:
        return None

    t_obs_end = target_window_days  # end of observed window in days since first review
    t_pred_end = state['first_review_dbc'] - midnight_utc_dbc  # end of prediction window (midnight UTC close day)

    F_obs = F_cluster(scores, t_obs_end)
    F_pred = F_cluster(scores, t_pred_end)

    if F_obs <= 0:
        return None
    V = state['observed_count'] / F_obs
    pred_future = V * (F_pred - F_obs)

    # Actual phase_1
    close_midnight = target_close.floor('D')
    mr = reviews_noon[reviews_noon['movie_slug']==slug]
    actual = int(((mr['estimated_timestamp'] >= snap_time) & (mr['estimated_timestamp'] < close_midnight)).sum())
    return {'slug': slug, 'V': V, 'F_obs': F_obs, 'F_pred': F_pred,
            'observed_count': state['observed_count'],
            'pred_future': float(pred_future), 'actual': actual,
            'err': float(pred_future) - actual}


# Evaluate on every target
rows = []
for slug in close_date_map:
    r = shape_scale_predict(slug)
    if r is not None:
        rows.append(r)
df = pd.DataFrame(rows)
print(f'Predictions for {len(df)} targets')

## Cohort-wide MAE comparison

In [ ]:
# Load prior predictions for comparison
# Baseline: weighted-KDE (midnight+noon)
def wkde_predict(slug):
    target_close = close_date_map[slug]
    midnight_utc_dbc = (target_close - target_close.floor('D')).total_seconds() / 86400
    snap_time = target_close.floor('D') - pd.Timedelta(days=SNAP_DAYS)
    snap_dbc_eff = (target_close - snap_time).total_seconds() / 86400

    obs = reviews_noon[(reviews_noon['movie_slug']==slug) & (reviews_noon['estimated_timestamp']<snap_time)]
    if len(obs) < 3:
        return None
    target_gap = gap_for_slug(slug)
    if target_gap is None:
        return None
    state = {
        'observed_critics': set(obs['reviewer_name']),
        'observed_count': len(obs),
        'first_review_dbc': float((target_close - obs['estimated_timestamp'].min()).total_seconds() / 86400),
    }
    if state['first_review_dbc'] < snap_dbc_eff + 1.0:
        return None
    tw = state['first_review_dbc'] - snap_dbc_eff
    scores = combined_score_with_scores(
        slug, target_gap, state['observed_critics'], tw,
        k=SHIP_N_TRAINING, alpha=SHIP_ALPHA, sigma_gap=SHIP_SIGMA_GAP,
    )
    if len(scores) < 5:
        return None
    try:
        profiles = build_weighted_critic_profiles(reviews_noon, close_date_map, scores, verbose=False)
        if len(profiles.df) == 0:
            return None
        model = build_weighted_kde_lambda_model(profiles, bandwidth_floor=0.5, bandwidth_ceiling=0.7)
        pred = predict_window_custom(
            model, dbc_from=snap_dbc_eff, dbc_to=midnight_utc_dbc,
            observed_critics=state['observed_critics'],
            observed_count=state['observed_count'],
            first_review_dbc=state['first_review_dbc'],
        )
        return float(pred)
    except Exception:
        return None

df['wkde_pred'] = df['slug'].apply(wkde_predict)

df['abs_err_shape'] = df['err'].abs()
df['err_wkde'] = df['wkde_pred'] - df['actual']
df['abs_err_wkde'] = df['err_wkde'].abs()

# Quartile stratification by actual
df['q_actual'] = pd.qcut(df['actual'], q=4, labels=['Q1','Q2','Q3','Q4'], duplicates='drop')

def summarize(sub, label):
    if len(sub) == 0:
        return
    mae_shape = sub['abs_err_shape'].mean()
    mae_wkde = sub['abs_err_wkde'].mean()
    me_shape = sub['err'].mean()
    me_wkde = sub['err_wkde'].mean()
    delta = (mae_wkde - mae_shape) / mae_wkde * 100 if mae_wkde > 0 else 0
    print(f'  {label:42s}  n={len(sub):3d}  wkde_MAE={mae_wkde:6.2f}  shape_MAE={mae_shape:6.2f}  delta={delta:+6.1f}%  (me: {me_wkde:+.2f} → {me_shape:+.2f})')

print('Shape×Scale vs weighted-KDE (positive delta = shape better):\n')
summarize(df, 'Full cohort')
print()
for q in ['Q1','Q2','Q3','Q4']:
    sub = df[df['q_actual'] == q]
    if len(sub):
        lo, hi = int(sub['actual'].min()), int(sub['actual'].max())
        summarize(sub, f'{q} (actual [{lo}, {hi}])')

## H/m subset

In [ ]:
hm_rows = df[df['slug'].isin(HM)].copy()
print('H/m per-target predictions:\n')
cols = ['slug', 'observed_count', 'V', 'F_obs', 'actual', 'wkde_pred', 'pred_future', 'err_wkde', 'err']
print(hm_rows[cols].rename(columns={'pred_future':'shape_pred', 'err':'shape_err'}).to_string(index=False, float_format='%.2f'))
print()
summarize(hm_rows, 'H/m aggregate (5 movies)')

## Cohort comparison summary

Bring in Ridge from the prior notebook for a 3-way comparison.

In [ ]:
# Compute Ridge predictions inline for comparison
# Need the features from earlier
from _helpers import critic_activity_counts, observed_review_stats
activity = critic_activity_counts()

def extract_features(slug):
    target_close = close_date_map[slug]
    snap_time = target_close.floor('D') - pd.Timedelta(days=SNAP_DAYS)
    snap_dbc_eff = (target_close - snap_time).total_seconds() / 86400
    close_midnight = target_close.floor('D')

    mr_all = reviews_noon[reviews_noon['movie_slug']==slug]
    obs = mr_all[(mr_all['estimated_timestamp'] < snap_time) & (mr_all['estimated_timestamp'] < target_close)]
    if len(obs) < 3:
        return None
    first_review_ts_target = obs['estimated_timestamp'].min()
    first_review_dbc = (target_close - first_review_ts_target).total_seconds() / 86400
    obs_window_days = first_review_dbc - snap_dbc_eff
    if obs_window_days <= 0:
        return None
    target_gap = gap_for_slug(slug)
    if target_gap is None:
        return None
    stats = observed_review_stats(slug, first_review_ts_target, obs_window_days, activity)
    last_day_start = snap_time - pd.Timedelta(days=1)
    rate_last_day = ((obs['estimated_timestamp'] >= last_day_start) & (obs['estimated_timestamp'] < snap_time)).sum()
    first_day_end = first_review_ts_target + pd.Timedelta(days=1)
    rate_first_day = ((obs['estimated_timestamp'] >= first_review_ts_target) & (obs['estimated_timestamp'] < first_day_end)).sum()
    actual = int(((mr_all['estimated_timestamp'] >= snap_time) & (mr_all['estimated_timestamp'] < close_midnight)).sum())
    return {
        'slug': slug, 'observed_count': len(obs), 'first_review_dbc': first_review_dbc,
        'target_gap': target_gap, 'observed_rate': len(obs)/obs_window_days,
        'rate_last_day': int(rate_last_day), 'rate_first_day': int(rate_first_day),
        'top_critic_frac': stats['top_critic_frac'], 'pub_diversity': stats['pub_diversity'],
        'pub_entropy': stats['pub_entropy'], 'low_activity_frac': stats['low_activity_frac'],
        'actual': actual,
    }

feat_rows = [extract_features(s) for s in close_date_map]
feat_rows = [r for r in feat_rows if r is not None]
feat = pd.DataFrame(feat_rows)

FEATURES = ['observed_count','first_review_dbc','target_gap','observed_rate',
            'rate_last_day','rate_first_day','top_critic_frac','pub_diversity','pub_entropy','low_activity_frac']

# Ridge: train on cohort minus h/m, predict all
cohort_feat = feat[~feat['slug'].isin(HM)]
X_c = cohort_feat[FEATURES].values
y_c = cohort_feat['actual'].values

# 5-fold CV on cohort for cohort predictions
kf = KFold(n_splits=5, shuffle=True, random_state=42)
ridge_cohort_preds = np.zeros(len(cohort_feat))
for train_idx, test_idx in kf.split(X_c):
    m = Ridge(alpha=10.0)
    m.fit(X_c[train_idx], y_c[train_idx])
    ridge_cohort_preds[test_idx] = m.predict(X_c[test_idx])

# For h/m: train on all cohort (no CV, since h/m is holdout)
ridge_all = Ridge(alpha=10.0)
ridge_all.fit(X_c, y_c)
hm_feat = feat[feat['slug'].isin(HM)]
ridge_hm_preds = ridge_all.predict(hm_feat[FEATURES].values)

# Merge ridge preds into main df
ridge_pred_map = dict(zip(cohort_feat['slug'].values, ridge_cohort_preds))
for slug, p in zip(hm_feat['slug'].values, ridge_hm_preds):
    ridge_pred_map[slug] = p
df['ridge_pred'] = df['slug'].map(ridge_pred_map)
df['err_ridge'] = df['ridge_pred'] - df['actual']
df['abs_err_ridge'] = df['err_ridge'].abs()

# 3-way summary
print('3-way comparison (cohort, no h/m contamination via CV/holdout):\n')
print(f'  {"scope":16s}  {"n":>4s}  {"wkde_MAE":>9s}  {"ridge_MAE":>10s}  {"shape_MAE":>10s}')
cohort_only = df[~df['slug'].isin(HM)]
print(f'  {"Cohort (no h/m)":16s}  {len(cohort_only):>4d}  {cohort_only["abs_err_wkde"].mean():>9.2f}  {cohort_only["abs_err_ridge"].mean():>10.2f}  {cohort_only["abs_err_shape"].mean():>10.2f}')
hm_only = df[df['slug'].isin(HM)]
print(f'  {"H/m":16s}  {len(hm_only):>4d}  {hm_only["abs_err_wkde"].mean():>9.2f}  {hm_only["abs_err_ridge"].mean():>10.2f}  {hm_only["abs_err_shape"].mean():>10.2f}')
print()
print('Mean-err (bias):')
print(f'  {"Cohort (no h/m)":16s}  wkde: {cohort_only["err_wkde"].mean():+6.2f}   ridge: {cohort_only["err_ridge"].mean():+6.2f}   shape: {cohort_only["err"].mean():+6.2f}')
print(f'  {"H/m":16s}  wkde: {hm_only["err_wkde"].mean():+6.2f}   ridge: {hm_only["err_ridge"].mean():+6.2f}   shape: {hm_only["err"].mean():+6.2f}')

print('\nPer-target h/m:')
print(f'  {"target":32s}  {"actual":>6s}  {"wkde":>8s}  {"ridge":>8s}  {"shape":>8s}  {"V":>7s}  {"F_obs":>6s}')
for _, r in hm_only.iterrows():
    print(f'  {r["slug"]:32s}  {r["actual"]:>6d}  {r["wkde_pred"]:>8.2f}  {r["ridge_pred"]:>8.2f}  {r["pred_future"]:>8.2f}  {r["V"]:>7.1f}  {r["F_obs"]:>6.3f}')

## Decision

Read MAE across cohort + h/m AND per-target h/m:
- Shape wins cohort + h/m MAE + h/m calibration → ship as primary.
- Shape wins cohort but loses h/m → tradeoff per use case.
- Ridge still best → shape model didn't materialize the expected gains.
- All three similar → we're data-limited, architecture choice is second-order.